## Шифрование при помощи RSA-алгоритма

In [ ]:
import random

class RSAAlgorithm:
    """
    Алгоритм RSA, позволяющий зашифровывать и расшифровывать текст
    при помощи методов encode() и decode().

    Берет случайные простые числа p и q в диапазоне 10^16 - 10^17,
    а также случайное e, такое что НОД(e, (p - 1) * (q - 1)) = 1.
    Реализован через собственный алфавит.
    """
    def __init__(self, show_variables=False) -> None:
        self.seed_value = 777 # секретное число для генерации случайных чисел
        random.seed(self.seed_value)
        self.p, self.q = self._get_rand_prime(), self._get_rand_prime() # p, q
        self.n = self.p * self.q # n = p * q
        self.phi = (self.p - 1) * (self.q - 1) # находим функцию Эйлера phi(n)
        self.e = self._get_rand_coprime(self.phi) # определяем e: НОД(e, phi) = 1
        self.d = self._get_inverse(self.e, self.phi) # находим d как обратный к e. d*e = 1 (mod phi(n))
        self.keys = self._set_keys() # соответствия букв и их кодов
        

        if show_variables:
            print("p = {}, q = {}, n = {}, phi = {}, e = {}, d = {}, keys = {}"
                  .format(self.p, self.q, self.n, self.phi, self.e, self.d, self.keys))

    def encode(self, text):
        """Зашифровывает текст и возвращает список кодов."""    
        res = []
        for char in text:
            # E(a) = a^e (mod n)
            code = self._pow(self.keys[char], self.e, self.n)
            res.append(code)
        return res
    
    def decode(self, codes):
        """Расшифровывает коды и возвращает текст."""
        res = []
        for code in codes:
            # D(b) = b^d (mod n)
            decoded = self._pow(code, self.d, self.n)
            for char, code in self.keys.items():
                if code == decoded:
                    res.append(char)
                    break
        return ''.join(res)

    def _is_prime(self, n):
        """Проверяет, является ли число n простым."""
        if n <= 1: return False
        if n <= 3: return True
        
        if n % 2 == 0 or n % 3 == 0:
            return False
        
        # общий вид простых чисел, кроме 2 и 3: 6k +- 1
        i = 5
        while i * i <= n:
            if n % i == 0 or n % (i + 2) == 0:
                return False
            i += 6
            
        return True

    def _get_rand_prime(self):
        """Возвращает случайное простое число в диапазоне 10^16 - 10^17."""
        while True:
            num = random.randint(10**16, 10**17)
            if self._is_prime(num):
                return num
    
    def _get_rand_coprime(self, n):
        """Находит случайное взаимно простое число с n (1 < result < n)."""
        def find_gcd(a, b):
            """Находит НОД по алгоритму Евклида."""
            while b != 0:
                a = a % b    # находим остаток от деления
                a, b = b, a  # меняем местами a и b
            return a

        while True:
            num = random.randint(2, n - 1)
            if find_gcd(num, n) == 1:
                # если НОД(num, n) = 1, то num является взаимно простым
                return num
    
    def _get_inverse(self, e, phi):
        """Находит d, такое что de = 1 (mod phi), используя расширенный алгоритм Евклида.
        
        Основная идея:
            * de = 1 (mod phi) -> сущ. k, такое, что de - 1 = k * phi
            * de - k*phi = 1
            * если a = e, x = d, b = phi, y = -k, то получаем диафантово ур-ние ax + by = 1
            * находим x через расширенный алгоритм Евклида
        """
        def find_xy(a, b):
            """Возвращает решение (x, y) для ур-ния ax + by = 1."""
            if a == 0:
                return 0, 1
            else:
                y, x = find_xy(b % a, a)
                return x - (b // a) * y, y

        x, y = find_xy(e, phi)
        # x может быть отрицательным, поэтому берем остаток от phi
        return x % phi

    def _set_keys(self):
        """Ставит в соответствие буквам случайный код от 2 до n - 1."""
        import string
        # добавляем английские буквы, русские буквы, знаки препинания, кавычки, тире, пробелы, цифры 
        letters = (string.ascii_letters + "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдеёжзийклмнопрстуфхцчшщъыьэюя" + 
                   string.punctuation + "«»" + '—' + string.whitespace + string.digits)
        seen = set()
        keys = {}
        for l in letters:
            val = random.randint(2, self.n - 1)
            while val in seen:
                val = random.randint(2, self.n - 1)
            keys[l] = val
            seen.add(val)
        return keys
    
    def _pow(self, a, n, d):
        """Вычисляет a^n mod d"""
        res = 1
        while n > 0:
            if n % 2 == 1:
                res = (res * a) % d
            a = (a * a) % d
            n //= 2
        return res

In [2]:
rsa = RSAAlgorithm(show_variables=True)

p = 96684248571659713, q = 81695062901973539, n = 7898625768691785546172812438334307, phi = 7898625768691785367793500964701056, e = 6884662202556156168848070317528711, d = 1650649950915080414902565981196215, keys = {'a': 4385469416935969287292827263103347, 'b': 6121432708597088269357970626981445, 'c': 5707408895299022224008751959582224, 'd': 6282994645949862348582339560205344, 'e': 3882482736354557980019671083750546, 'f': 4507107723179500710487104787370556, 'g': 7307156972961825480789452007657468, 'h': 4109558454361469240476012803663469, 'i': 5077438294365802517961490017849635, 'j': 4964724985909689115659014333723549, 'k': 838655821799323640897002891941928, 'l': 3128503588583516786611092147703623, 'm': 5785390933197199111784248516316483, 'n': 1743936125386382427531011220815178, 'o': 4014261027553747841567332544737652, 'p': 7736994676192871633909416605285258, 'q': 6675220712969981966539015856334326, 'r': 3082736325274268089597155812548272, 's': 4493707615088856577206416508869449, 't': 2

In [3]:
text = "Привет!"

encoded = rsa.encode(text)
print("Зашифрованное сообщение:", encoded)

decoded = rsa.decode(encoded)
print("Расшифрованное сообщение:", decoded)

Зашифрованное сообщение: [332646575686135764317731295665326, 7226020655858541371671191564330919, 745582800692168940858505821676055, 2375227110216966924627012322301764, 4625981978796717213439686490276261, 6950132562331204591350639849016461, 6010760003391707989782598868328890]
Расшифрованное сообщение: Привет!


In [4]:
text = "Четные числа - питательны, а нечетные - просто вкусные"

encoded = rsa.encode(text)
print("Зашифрованное сообщение:", encoded)

decoded = rsa.decode(encoded)
print("Расшифрованное сообщение:", decoded)

Зашифрованное сообщение: [2783952084778655882786112881483453, 4625981978796717213439686490276261, 6950132562331204591350639849016461, 7509958910210550892361317293297671, 1160442044540063639359486235233491, 4625981978796717213439686490276261, 4353582096356705474216633275594820, 5694563235971549550971835204759326, 745582800692168940858505821676055, 950135377985336472177797596355263, 7865010764966657038885590889254353, 5990528641948580402590284554863765, 4353582096356705474216633275594820, 282082069937888912369984192042471, 4353582096356705474216633275594820, 527293320129422837457005242255284, 745582800692168940858505821676055, 6950132562331204591350639849016461, 5990528641948580402590284554863765, 6950132562331204591350639849016461, 4625981978796717213439686490276261, 7865010764966657038885590889254353, 6887771454937295077782525612318161, 7509958910210550892361317293297671, 1160442044540063639359486235233491, 992935643452857695331472531917369, 4353582096356705474216633275594820, 59905286

In [5]:
text = """Недавно мне захотелось узнать, какой была жизнь 100 000 лет назад.
Я забрался в воображаемую машину времени и перенесся в прошлое, чтобы взять интервью у Стэнли, доисторического невротика.
— Привет, Стэнли! Меня зовут Боб, и я тут недавно. У тебя найдется пара минут поговорить со мной?
— Не видишь, я занят? Я тороплюсь. Думаю, сейчас пойдет дождь,
а я забыл свою львиную шкуру. Ну, я могу простудиться, поднимется
температура. Чего ты хотел?
— Я хотел побольше узнать о том, как ты живешь.
— Я сделал что-то не так? Мне казалось, на прошлой неделе я погасил огонь — неужели что-то сгорело? Я стараюсь быть осторожным,
но никогда не знаешь наверняка.
— Нет, Стэнли. Мне просто хотелось узнать, как тебе живется.
— Как живется? Ну, во-первых, я не могу спать. Всю ночь глаз
не сомкнуть. Беспокоят тигры и волки в лесу. Мне говорят, что тут
безопасно, но я слышал такие истории, что волосы дыбом встают.
— Так у тебя бессонница?
— Не всегда. Но она сводит меня с ума. Я не могу спать, постоянно
повторяю себе: Стэнли, поспи хоть чуть-чуть! Ты ни на что не годен,
когда не высыпаешься.
— Стэнли, а расскажи про местность, в которой ты живешь?
— Она опасная. Осторожнее на мосту. Не знаю, какой растяпа
его мастерил, но один неверный шаг — и можешь упасть. А потом
попробуй найди врача!
— Так ты боишься высоты?
— Эй, да только дураки ее не боятся. Высота может убить. Не
умрешь от падения, так утонешь. В прошлом месяце двое этаких «настоящих мужиков» срубили бревно и попытались переплыть на нем
на другой остров. Утонули! А я предупреждал. Но нет, они думали,
что и сами все прекрасно знают. Говорили: «Стэнли, ты всегда волнуешься — такой пессимист! Такой настрой вреден для пищеварения».
И вот они теперь на дне морском, а я разговариваю с тобой. И кто из
нас умнее?
— Видимо, ты, Стэнли.
— Вот именно. Правда, у меня иногда-таки случается несварение.
Надеюсь, ничего серьезного. Но все равно сходил к доктору. Может,
ты его видел, у него змея вокруг головы.
— Нет, я тут совсем недавно, еще не видел его.
— Ну так вот, он дал мне такую отвратительную микстуру — для
моего живота. Это должно было меня успокоить. Но меня вытошнило.
Я просто боюсь, что со мной что-то серьезно не так. А что, в том году
на юге отсюда целое племя отравилось и вымерло. Я мою все —
все! — прежде чем класть в рот. Но я не уверен, что и воду здесь
пить безопасно. С тех пор как тут поселилось столько народу, мне
кажется, вода загрязнилась. Кто-то считает меня фанатиком, но зато
я еще ни разу не отравился.
— Ага, ты все еще жив, Стэнли. А как ты относишься к чужакам?
— Я им не доверяю. Кто их знает. Они могут атаковать меня, забрать мою Сару и детей, выгнать меня из племени. Так что я стараюсь
не привлекать к себе внимание. Не смотрю им в глаза. Говорю мягко.
Не закатываю сцен. Как там было, в пословице... Тише едешь, будешь
жив? Вот я и веду себя тихо. Можно сказать, я немного стеснительный.
— А чего ты боишься?
— Ну, помимо всего прочего... я боюсь, что меня убьют. К этому
все сводится.
— С тобой уже случалось что-то подобное?
— В прошлом месяце — до сих пор не могу выкинуть это из головы — я видел чужаков, иностранцев. На них были шкуры других
животных, и волосы у них были другого цвета. Это было ужасно.
Я видел, как они схватили и связали моего дядю Гарри, а потом сварили его и съели. Страшная сцена. Слава богу, я к тому моменту уже
бежал без оглядки — но выкинуть эту картинку из головы просто не
могу. Посреди ночи просыпаюсь от его криков. А ведь следующим
могу оказаться я.
— Звучит ужасно. Так ты боишься, что с тобой произойдет то же,
что и с твоим дядей Гарри?
— Да, но, с позволения сказать, я поумнее его. Дядя Гарри был тот
еще балабол, всегда говорил: «Стэнли, ты такой пессимист. Чего ты
так переживаешь?» При всем уважении к покойнику — ужином стал
он, а у меня всего лишь несварение.
— Тебе, должно быть, трудно все время столько волноваться.
— Трудно? Да это просто ужасно. Два года назад у меня стали случаться «атаки». Например, иду я по полю, как вдруг сердце начинает
очень быстро биться. Солнце светит, нигде никаких львов. А я вдруг
так испугался, что у меня сердечный приступ, аж голова закружилась.
Едва перевел дыхание.
— И что было потом?
— Мне повезло, Сара была рядом — благослови Бог ее душу. Она
взяла меня под руку и отвела в ближайшую пещеру. У меня так кружилась голова. И потом целых три месяца мне было очень страшно
выходить в поле. Боялся, что снова будет атака.
— И как ты это преодолел?
— Да никак. Хотя в последнее время ничего такого не было. Думаю, помогло то, что Сара заставила меня, — а она может быть очень
настойчивой, по правде говоря, — заставила пойти собрать в поле
клубнику, пока птицы ее не склевали. Я был голоден и подумал: «Сара
со мной, если случится атака, она мне поможет». Думаю, благодаря
этому я и собрал клубнику. Но я все равно переживаю: вдруг атаки
повторятся.
— Ты вообще тревожный?
— Конечно. Я волнуюсь, достаточно ли еды. Или вдруг Сара убежит от меня с кем-то, кто сильнее. Волнуюсь о детях — вдруг попадут
в беду. И всегда волнуюсь за свое здоровье.
— Это, наверное, тяжело.
— Осторожнее!!!
— В чем дело, Стэнли?
— А, нет, все в порядке. Мне показалось, что вон та веточка — это
змея. Змеи ядовиты, так что будь осторожнее.
— Спасибо, что уделил мне время, Стэнли.
— Всегда пожалуйста. Надеюсь, я не был слишком пессимистичен.
Все говорят: «Стэнли, ты такой пессимист. Всего боишься. Всегда
ожидаешь худшего». Может, они и правы. Может, я теряю рассудок.
Погоди-ка... а накрыл ли я еду камнем? Вдруг кто-то будет проходить
мимо — вдруг медведь залезет в пещеру, найдет еду — что если
запах еды привлечет медведя в пещеру, — потому что я не накрыл
еду, — и он убьет всю мою семью?! И я буду виноват, потому что был
невнимательным. Мне кажется, я клал камень но я не уверен...
— Стэнли, мне пора.
— Может, я и накрыл еду камнем. Надо пойти проверить. Но
если я пойду, я пропущу встречу с Сидом. Он разозлится. А когда он
разозлится, наговорит всем, какой я безответственный. И что тогда?

Ладно, признаюсь, я выдумал Стэнли из каменного века. Но вы
понимаете, что я имел в виду. Какова была жизнь с эволюционной
точки зрения, с точки зрения наших предков, живших сотни тысяч
лет назад? Какие качества были нужны людям, чтобы справляться
с выпавшими на их долю испытаниями, — и как это связано с нами
сегодня. Эволюционная психология уже какое-то время собирает
эту мозаику. И тревога — один из ее кусочков. Так давайте серьезно
обсудим, как тревога стала частью жизни людей.

Фрагмент взят из книги Роберта Лихи "Свобода от тревоги"
"""

encoded = rsa.encode(text)
print("Зашифрованное сообщение:", encoded)

decoded = rsa.decode(encoded)
print("Расшифрованное сообщение:", decoded)

Зашифрованное сообщение: [3137043544577853631713571398014296, 4625981978796717213439686490276261, 3370221491651954964430437038601425, 5990528641948580402590284554863765, 2375227110216966924627012322301764, 7509958910210550892361317293297671, 2702257727855318848235348677831900, 4353582096356705474216633275594820, 33859264527946554369619620643952, 7509958910210550892361317293297671, 4625981978796717213439686490276261, 4353582096356705474216633275594820, 22588506065458117746611924502624, 5990528641948580402590284554863765, 44534591858707937884698041732638, 2702257727855318848235348677831900, 6950132562331204591350639849016461, 4625981978796717213439686490276261, 7865010764966657038885590889254353, 2702257727855318848235348677831900, 950135377985336472177797596355263, 6887771454937295077782525612318161, 4353582096356705474216633275594820, 5397154627941857186079730379325789, 22588506065458117746611924502624, 7509958910210550892361317293297671, 5990528641948580402590284554863765, 69501325623